# AI Assignment — General-Purpose Classical Reasoning System

**Pipeline:** Natural-language knowledge base → internal representation (facts + rules) → classical reasoning → `True` / `False`

| Component | What it does |
|---|---|
| §1 Representation | Literals `(sign, predicate, args)`, Horn rules with variables |
| §2 NL → logic parser | Rule-based, deterministic parser for RuleTaker-style language (facts, `If … then …`, quantified rules, negation) |
| §3 Forward chaining | Semi-naive bottom-up inference with unification; negation-as-failure via the alternating fixpoint (well-founded semantics); proof traces |
| §4 Backward chaining | SLD resolution with loop check, depth limit and NAF (used for comparison / efficiency experiments) |
| §5 FOL resolution | FOL parser → CNF/Skolemisation → set-of-support resolution (used when first-order annotations are available, e.g. FOLIO) |
| §6 Causal-model reasoner | Deterministic structural causal model + abduction/intervention for CounterBench-style counterfactual queries |
| §7 Unified `Reasoner` | Chooses the pipeline for each example and returns `True`/`False` |

**No generative AI is used anywhere**: the final decision for every query is made by the classical engines above.

In [1]:
import re, time, json, csv, os, sys, random, itertools, heapq
from collections import defaultdict

CONFIG = dict(
    mode='cwa',            # 'cwa' (closed world, negation-as-failure) or 'owa' (explicit negation). Chosen on dev data below.
    engine='fc',           # 'fc' forward chaining | 'bc' backward chaining
    fol_timeout=5.0,       # seconds per resolution call
)

## 1. Internal representation
* **Term**: a constant (`'bob'`) or a variable (`'?x'`).
* **Literal**: `(sign, predicate, args)`; `sign=False` means the literal is negated.
* **Rule**: Horn clause `body₁ ∧ … ∧ bodyₙ → head` (body literals may be negated).
* **Facts**: ground literals.

Unary predicates encode properties (`young(bob)`), binary predicates encode relations (`chase(cat, dog)`).

In [2]:
def is_var(t):
    return isinstance(t, str) and t[:1] == '?'

def atom_str(pred, args):
    return f"{pred}({', '.join(map(str, args))})" if args else pred

def lit_str(l):
    s, p, a = l
    return ('' if s else '¬') + atom_str(p, a)

class Rule:
    """Horn rule: body is a list of literals, head a single literal."""
    __slots__ = ('body', 'head', 'text')
    def __init__(self, body, head, text=''):
        self.body, self.head, self.text = body, head, text
    def __repr__(self):
        return (' ∧ '.join(lit_str(l) for l in self.body) or '⊤') + ' → ' + lit_str(self.head)

## 2. Natural language → logic (rule-based parser)
Handled sentence shapes (case-insensitive):
* facts: `Bob is young.` · `Bob is not big.` · `The cat chases the bald eagle.` · `The dog does not like the cat.` · `Bob is big and red.`
* rules: `If someone is young, then they are smart.` · `If something is smart and kind then it is successful.` · `If something likes the dog and it is red then it is nice.`
* generic rules: `All red things are big.` · `Big, red things are nice.` · `Round people are not kind.` · `Everyone is nice.`

Anything that does not match a template is turned into a 0-ary *propositional* atom (so simple propositional text still works),
and every such sentence is counted in `Theory.unparsed` so parsing coverage can be reported.

In [3]:
COPS = {'is', 'are', 'was', 'were'}
VARWORDS = {'something', 'someone', 'somebody', 'it', 'they', 'he', 'she', 'anything', 'anyone'}
HEADNOUNS = {'things', 'thing', 'people', 'person', 'persons', 'ones'}
NOT_VERBS = {'not', 'and', 'is', 'are', 'was', 'were', 'then', 'if', 'the', 'a', 'an'}

def tok(s):
    s = s.lower()
    s = re.sub(r"\b(does|do|did|is|are|was|were|has|have|can|will)n't\b", r"\1 not", s)
    s = s.replace("won't", 'will not')
    return re.findall(r"[a-z0-9_\-]+|,", s)

def lemma(w):
    if w.endswith('ies') and len(w) > 4: return w[:-3] + 'y'
    if re.search(r'(sses|ches|shes|xes|zes)$', w): return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3: return w[:-1]
    return w

def _entity(t):
    """token list -> entity name, or None if the tokens are not a (short) entity phrase"""
    if not t: return None
    if t[0] in ('the', 'a', 'an'):
        r = t[1:]
        return ' '.join(r) if 1 <= len(r) <= 3 else None
    return t[0] if len(t) == 1 and t[0] not in VARWORDS else None

def _subj_cands(t):
    if not t: return []
    h = t[0]
    if h in VARWORDS: return [('?x', t[1:])]
    if h in ('the', 'a', 'an'):
        return [(' '.join(t[1:1 + k]), t[1 + k:]) for k in (1, 2, 3) if len(t) > 1 + k]
    return [(h, t[1:])]

def _split_adj(r):
    """['big','and','not','red'] -> [(False,['big']), (True,['red'])]"""
    parts, cur = [], []
    for w in r + ['and']:
        if w in ('and', ','):
            if cur:
                neg = cur[0] == 'not'
                words = cur[1:] if neg else cur
                if words: parts.append((neg, words))
            cur = []
        else:
            cur.append(w)
    return parts

def parse_atom(t):
    """token list -> list of literals (a conjunction) or None"""
    for subj, rest in _subj_cands(t):
        if not rest: continue
        if rest[0] in COPS:
            r = rest[1:]
            if r[:1] in (['a'], ['an']) and len(r) > 1: r = r[1:]
            parts = _split_adj(r)
            if parts:
                return [(not neg, '_'.join(w), (subj,)) for neg, w in parts]
            continue
        r, neg = rest, False
        if r[:2] in (['does', 'not'], ['do', 'not'], ['did', 'not']):
            neg, r = True, r[2:]
        if len(r) >= 2 and r[0] not in NOT_VERBS:
            obj = _entity(r[1:])
            if obj is not None:
                return [(not neg, lemma(r[0]), (subj, obj))]
    return None

def prop_lit(t):
    neg = any(w in ('not', 'no', 'never') for w in t)
    words = [w for w in t if w not in ('not', 'no', 'never', ',')]
    return (not neg, '_'.join(words) or 'empty', ())

def parse_conj(t):
    """conjunction of literals; continuation parts without subject inherit the previous subject."""
    parts, cur = [], []
    for w in t:
        if w in ('and', ','):
            if cur: parts.append(cur); cur = []
        else:
            cur.append(w)
    if cur: parts.append(cur)
    if not parts: return None
    lits, last = [], None
    for p in parts:
        a = parse_atom(p)
        if a is None and last is not None and (p[0] in ('does', 'do') or (len(p) >= 3 and p[1] == 'the')):
            a = parse_atom(['it'] + p)                 # "... and likes the cat"
            if a is not None: a = [(s, pr, (last,) + ar[1:]) for s, pr, ar in a]
        if a is not None:
            lits += a; last = a[-1][2][0]; continue
        if last is not None:                           # adjective continuation: "smart and kind"
            neg = p[0] == 'not'
            w = p[1:] if neg else p
            if w:
                lits.append((not neg, '_'.join(w), (last,))); continue
        return None
    return lits

def parse_conj_or_prop(t):
    r = parse_conj(t)
    if r is not None: return r, True
    parts, cur = [], []
    for w in t:
        if w in ('and', ','):
            if cur: parts.append(cur); cur = []
        else: cur.append(w)
    if cur: parts.append(cur)
    return [prop_lit(p) for p in parts], False

def parse_generic(t):
    """'All big, red things are nice' / 'Everyone is kind' -> (body, heads) or None"""
    if not t: return None
    if t[0] in ('everyone', 'everything', 'everybody') and len(t) > 1:
        h = parse_atom(['it'] + t[1:])
        return ([], h) if h else None
    start = 1 if t[0] in ('all', 'every', 'each', 'any') else 0
    k = next((i for i in range(start, len(t)) if t[i] in HEADNOUNS), None)
    if k is None or k + 1 >= len(t): return None
    pre = t[start:k]
    if any(w in COPS or w in ('the', 'a', 'an', 'if', 'then') for w in pre): return None
    heads = parse_atom(['it'] + t[k + 1:])
    if not heads: return None
    body = [(not neg, '_'.join(w), ('?x',)) for neg, w in _split_adj(pre)]
    return body, heads

class Theory:
    def __init__(self):
        self.facts, self.rules, self.unparsed, self.domain = [], [], [], set()

def parse_sentence(s, th):
    s0 = s.strip().rstrip('.!?;').strip()
    if not s0: return
    t = tok(s0)
    if not t: return
    if t[0] == 'if' and 'then' in t:
        i = t.index('then')
        body, ok1 = parse_conj_or_prop(t[1:i])
        heads, ok2 = parse_conj_or_prop(t[i + 1:])
        if not (ok1 and ok2): th.unparsed.append(s0)
        for h in heads: th.rules.append(Rule(list(body), h, s0))
        return
    g = parse_generic(t)
    if g is not None:
        body, heads = g
        for h in heads: th.rules.append(Rule(list(body), h, s0))
        return
    a = parse_atom(t)
    if a is None:
        th.unparsed.append(s0); a = [prop_lit(t)]
    for l in a:
        if any(is_var(x) for x in l[2]):               # existential statements are not supported
            th.unparsed.append(s0); continue
        th.facts.append((l, s0))

def split_sentences(text):
    return [x.strip() for x in re.split(r'(?<=[.!?])\s+|\n+', (text or '').strip()) if x.strip()]

def parse_theory(text):
    th = Theory()
    for s in split_sentences(text): parse_sentence(s, th)
    for (_, _, a), _ in th.facts: th.domain.update(a)
    for r in th.rules:
        for l in r.body + [r.head]: th.domain.update(x for x in l[2] if not is_var(x))
    return th

def parse_query(q):
    t = tok(q.strip().rstrip('.!?').strip())
    lits, _ = parse_conj_or_prop(t)
    return lits

## 3. Forward chaining (bottom-up, semi-naive) with negation
* **Positive Horn part**: semi-naive evaluation — each round joins every rule so that at least one body literal matches a *newly* derived fact (unification by matching ground tuples against rule patterns).
* **Closed-world mode (`cwa`)**: `not p` in a rule body / query means *p cannot be derived* (negation-as-failure). Implemented with the **alternating fixpoint**: an under-estimate `T` (true facts) and an over-estimate `O` (possibly true facts) are refined until stable — this equals the perfect model on stratified theories and the well-founded model otherwise.
* **Open-world mode (`owa`)**: `¬p` is an ordinary derivable literal (stored as predicate `~p`); a query is *True* if derived, *False* if its negation is derived, otherwise *Unknown*.
* Every derived fact stores the rule instance that produced it, so a **proof trace** can be printed.

In [4]:
class CRule:
    __slots__ = ('pos', 'neg', 'head', 'text', 'vars')
    def __init__(self, pos, neg, head, text):
        self.pos, self.neg, self.head, self.text = pos, neg, head, text
        vs = []
        for _, a in pos + neg + [head]:
            for x in a:
                if is_var(x) and x not in vs: vs.append(x)
        self.vars = vs

def compile_theory(th, mode):
    facts, crules = defaultdict(set), []
    for (s, p, a), _ in th.facts:
        if mode == 'owa': facts[p if s else '~' + p].add(a)
        elif s: facts[p].add(a)                       # CWA: negative facts carry no information
    for r in th.rules:
        if mode == 'owa':
            f = lambda l: ((l[1] if l[0] else '~' + l[1]), l[2])
            crules.append(CRule([f(l) for l in r.body], [], f(r.head), r.text))
        else:
            if not r.head[0]: continue                # CWA: negative heads are meaningless
            pos = [(l[1], l[2]) for l in r.body if l[0]]
            neg = [(l[1], l[2]) for l in r.body if not l[0]]
            crules.append(CRule(pos, neg, (r.head[1], r.head[2]), r.text))
    return dict(facts), crules

def _subst(args, sub):
    return tuple(sub.get(a, a) if is_var(a) else a for a in args)

def _match(pattern, tup, sub):
    s = None
    for a, v in zip(pattern, tup):
        if is_var(a):
            cur = (s if s is not None else sub).get(a)
            if cur is None:
                if s is None: s = dict(sub)
                s[a] = v
            elif cur != v: return None
        elif a != v: return None
    return s if s is not None else sub

def _complete(sub, rule, domain):
    unb = [v for v in rule.vars if v not in sub]
    if not unb:
        yield sub; return
    for combo in itertools.product(domain, repeat=len(unb)):
        s = dict(sub); s.update(zip(unb, combo)); yield s

def _join(pos, skip, sub, full):
    idx = [j for j in range(len(pos)) if j != skip]
    def rec(k, s):
        if k == len(idx):
            yield s; return
        p, a = pos[idx[k]]
        for tup in full.get(p, ()):
            s2 = _match(a, tup, s)
            if s2 is not None: yield from rec(k + 1, s2)
    yield from rec(0, sub)

def lfp(facts, crules, domain, neg_ok=None, want_prov=False):
    """least fixpoint; neg_ok(pred,args) -> True iff 'not pred(args)' holds. Returns (model, provenance)."""
    full = {p: set(s) for p, s in facts.items()}
    prov = {} if want_prov else None
    def emit(rule, sub, out):
        for p, a in rule.neg:
            if not neg_ok(p, _subst(a, sub)): return
        hp, ha = rule.head[0], _subst(rule.head[1], sub)
        if ha in full.get(hp, ()) or ha in out.get(hp, ()): return
        out.setdefault(hp, set()).add(ha)
        if prov is not None: prov[(hp, ha)] = (rule, dict(sub))
    out = {}
    for rule in crules:
        if not rule.pos:
            for sub in _complete({}, rule, domain): emit(rule, sub, out)
    delta = {p: set(s) for p, s in facts.items()}
    for p, s in out.items():
        full.setdefault(p, set()).update(s); delta.setdefault(p, set()).update(s)
    prules = [r for r in crules if r.pos]
    while delta:
        out = {}
        for rule in prules:
            for i, (pi, ai) in enumerate(rule.pos):
                d = delta.get(pi)
                if not d: continue
                for tup in d:
                    s0 = _match(ai, tup, {})
                    if s0 is None: continue
                    for s in _join(rule.pos, i, s0, full):
                        for s2 in _complete(s, rule, domain): emit(rule, s2, out)
        delta = {}
        for p, s in out.items():
            full.setdefault(p, set()).update(s); delta[p] = s
    return full, prov

def solve_fc(facts, crules, domain, mode, want_prov=False):
    """returns (T, O, prov): true set, possibly-true set (CWA); OWA returns (T, T, prov)."""
    if mode == 'owa' or not any(r.neg for r in crules):
        T, prov = lfp(facts, crules, domain, None, want_prov); return T, T, prov
    T = {}
    for _ in range(200):
        O, _p = lfp(facts, crules, domain, lambda p, a, T=T: a not in T.get(p, ()))
        T2, prov = lfp(facts, crules, domain, lambda p, a, O=O: a not in O.get(p, ()), want_prov)
        if T2 == T: break
        T = T2
    return T2, O, prov

def holds(model, lit, mode):
    T, O = model
    s, p, a = lit
    if mode == 'owa':
        return a in T.get(p if s else '~' + p, ())
    return (a in T.get(p, ())) if s else (a not in O.get(p, ()))

def explain_fc(prov, facts, pred, args, depth=0, seen=None, mode='cwa'):
    """human-readable derivation tree"""
    seen = set() if seen is None else seen
    pad = '    ' * depth
    name = atom_str(pred, args)
    if args in facts.get(pred, ()): return [f'{pad}{name}   [given fact]']
    if (pred, args) not in prov: return [f'{pad}{name}   [no derivation]']
    rule, sub = prov[(pred, args)]
    lines = [f'{pad}{name}   ⇐ "{rule.text}"']
    if (pred, args) in seen: return lines
    seen = seen | {(pred, args)}
    for p, a in rule.pos:
        lines += explain_fc(prov, facts, p, _subst(a, sub), depth + 1, seen, mode)
    for p, a in rule.neg:
        lines.append(f'{pad}    not {atom_str(p, _subst(a, sub))}   [cannot be derived → NAF]')
    return lines

## 4. Backward chaining (goal-directed SLD resolution)
Query-driven alternative: to prove a ground goal, unify it with rule heads and recursively prove the body.
Loop check on the ancestor goals, depth limit, a step budget, and NAF for negated goals (fresh ancestor set).

In [5]:
class BackwardChainer:
    def __init__(self, facts, crules, domain, max_depth=40, max_steps=200000):
        self.facts, self.domain = facts, list(domain)
        self.by_head = defaultdict(list)
        for r in crules:
            r2 = r
            self.by_head[r.head[0]].append(r2)
        self.max_depth, self.max_steps, self.steps = max_depth, max_steps, 0
        self.cache = {}

    def prove(self, pred, args, anc=frozenset(), depth=0):
        if args in self.facts.get(pred, ()): return True
        if (pred, args) in self.cache: return True
        if (pred, args) in anc or depth > self.max_depth: return False
        self.steps += 1
        if self.steps > self.max_steps: return False
        anc2 = anc | {(pred, args)}
        for rule in self.by_head.get(pred, ()):
            sub = _match(rule.head[1], args, {})
            if sub is None: continue
            lits = [(True, p, a) for p, a in rule.pos] + [(False, p, a) for p, a in rule.neg]
            for _ in self._body(lits, sub, anc2, depth + 1):
                self.cache[(pred, args)] = True
                return True
        return False

    def _body(self, lits, sub, anc, depth):
        if not lits:
            yield sub; return
        (sg, p, a), rest = lits[0], lits[1:]
        ga = _subst(a, sub)
        unb = list(dict.fromkeys(x for x in ga if is_var(x)))
        combos = itertools.product(self.domain, repeat=len(unb)) if unb else [()]
        for combo in combos:
            s2 = dict(sub); s2.update(zip(unb, combo))
            g = _subst(a, s2)
            ok = self.prove(p, g, anc, depth) if sg else (not self.prove(p, g, frozenset(), 0))
            if ok: yield from self._body(rest, s2, anc, depth)

## 5. First-order logic: CNF conversion + resolution refutation
Used when a first-order annotation of the premises is available (FOLIO provides `premises-FOL` / `conclusion-FOL`).
1. Parse `∀ ∃ ¬ ∧ ∨ → ↔ ⊕` formulas.  2. Eliminate `→ ↔ ⊕`, push negations (NNF).  3. Skolemise existentials.  4. Distribute `∨` over `∧` → clauses.
5. **Resolution with unification (occurs check), factoring, tautology/duplicate removal, set-of-support, smallest-clause-first** within a time limit.

*Entailment test*: premises ∪ {¬conclusion} unsatisfiable ⇒ **True**; premises ∪ {conclusion} unsatisfiable ⇒ **False** (contradiction); otherwise **Uncertain**.

In [6]:
_FOL_TOK = re.compile(r"\s*(?:(∀|∃|¬|∧|∨|⊕|→|↔|\(|\)|,)|([A-Za-z_][A-Za-z0-9_\-']*|\d+))")

def _fol_norm(s):
    for a, b in [('<->', '↔'), ('->', '→'), ('&&', '∧'), ('||', '∨'), ('&', '∧'), ('|', '∨'), ('~', '¬'), ('!', '¬'),
                 ('⟷', '↔'), ('⇒', '→'), ('⟶', '→'), ('forall ', '∀'), ('exists ', '∃'), ('∀ ', '∀'), ('∃ ', '∃')]:
        s = s.replace(a, b)
    return s

def _fol_tokens(s):
    s = _fol_norm(s); pos, out = 0, []
    while pos < len(s):
        m = _FOL_TOK.match(s, pos)
        if not m:
            if s[pos:].strip() == '': break
            raise ValueError(f'FOL tokenisation error near: {s[pos:pos+15]!r}')
        out.append(m.group(1) or m.group(2)); pos = m.end()
    return out

class FOLParser:
    def __init__(self, s): self.t, self.i = _fol_tokens(s), 0
    def peek(self, k=0): return self.t[self.i + k] if self.i + k < len(self.t) else None
    def nxt(self):
        x = self.peek(); self.i += 1; return x
    def expect(self, x):
        if self.nxt() != x: raise ValueError(f'expected {x}')
    def parse(self):
        f = self.iff()
        if self.peek() is not None: raise ValueError('trailing tokens')
        return f
    def iff(self):
        l = self.imp()
        while self.peek() == '↔': self.nxt(); l = ('iff', l, self.imp())
        return l
    def imp(self):
        l = self.or_()
        if self.peek() == '→': self.nxt(); return ('imp', l, self.imp())
        return l
    def or_(self):
        l = self.xor()
        while self.peek() == '∨': self.nxt(); l = ('or', l, self.xor())
        return l
    def xor(self):
        l = self.and_()
        while self.peek() == '⊕': self.nxt(); l = ('xor', l, self.and_())
        return l
    def and_(self):
        l = self.un()
        while self.peek() == '∧': self.nxt(); l = ('and', l, self.un())
        return l
    def un(self):
        t = self.peek()
        if t == '¬': self.nxt(); return ('not', self.un())
        if t in ('∀', '∃'):
            self.nxt()
            if self.peek() is None or not re.match(r'[A-Za-z_]', self.peek()): raise ValueError('quantifier without variable')
            vs = [self.nxt().lower()]                      # first identifier is always the bound variable
            while self.peek() is not None and re.match(r'[A-Za-z_]', self.peek()) and self.peek(1) not in ('(', None, ')', ','):
                vs.append(self.nxt().lower())              # "∀x y P(x,y)" style lists
            body = self.un() if self.peek() == '(' else self.iff()
            return ('all' if t == '∀' else 'ex', tuple(vs), body)
        if t == '(':
            self.nxt(); f = self.iff(); self.expect(')'); return f
        if t is None or not re.match(r'[A-Za-z_0-9]', t): raise ValueError('unexpected token ' + str(t))
        name = self.nxt().lower()
        args = ()
        if self.peek() == '(':
            self.nxt(); a = [self.term()]
            while self.peek() == ',': self.nxt(); a.append(self.term())
            self.expect(')'); args = tuple(a)
        return ('pred', name, args)
    def term(self):
        t = self.nxt()
        if t is None or not re.match(r'[A-Za-z_0-9]', t): raise ValueError('bad term')
        name = t.lower()
        if self.peek() == '(':
            self.nxt(); a = [self.term()]
            while self.peek() == ',': self.nxt(); a.append(self.term())
            self.expect(')'); return (name,) + tuple(a)
        return name

def _elim(f):
    t = f[0]
    if t == 'pred': return f
    if t == 'not': return ('not', _elim(f[1]))
    if t in ('and', 'or'): return (t, _elim(f[1]), _elim(f[2]))
    if t == 'imp': return ('or', ('not', _elim(f[1])), _elim(f[2]))
    if t == 'iff':
        a, b = _elim(f[1]), _elim(f[2]); return ('and', ('or', ('not', a), b), ('or', ('not', b), a))
    if t == 'xor':
        a, b = _elim(f[1]), _elim(f[2]); return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    return (t, f[1], _elim(f[2]))

def _nnf(f, neg=False):
    t = f[0]
    if t == 'pred': return ('not', f) if neg else f
    if t == 'not': return _nnf(f[1], not neg)
    if t in ('and', 'or'):
        t2 = {'and': 'or', 'or': 'and'}[t] if neg else t
        return (t2, _nnf(f[1], neg), _nnf(f[2], neg))
    q = {'all': 'ex', 'ex': 'all'}[t] if neg else t
    return (q, f[1], _nnf(f[2], neg))

_uid = itertools.count(1)

def _sub_term(t, env):
    if isinstance(t, tuple): return (t[0],) + tuple(_sub_term(x, env) for x in t[1:])
    return env.get(t, t)

def _skolem(f, env, univ):
    t = f[0]
    if t == 'pred': return ('pred', f[1], tuple(_sub_term(a, env) for a in f[2]))
    if t == 'not': return ('not', _skolem(f[1], env, univ))
    if t in ('and', 'or'): return (t, _skolem(f[1], env, univ), _skolem(f[2], env, univ))
    env2, univ2 = dict(env), list(univ)
    for v in f[1]:
        if t == 'all':
            nv = f'?{v}_{next(_uid)}'; env2[v] = nv; univ2.append(nv)
        else:
            nm = f'sk{next(_uid)}'; env2[v] = (nm,) + tuple(univ) if univ else nm
    return _skolem(f[2], env2, univ2)

def _cnf(f, cap=4000):
    t = f[0]
    if t == 'pred': return [[(True, f[1], f[2])]]
    if t == 'not': return [[(False, f[1][1], f[1][2])]]
    a, b = _cnf(f[1], cap), _cnf(f[2], cap)
    if t == 'and': return a + b
    if len(a) * len(b) > cap: raise ValueError('CNF blow-up')
    return [x + y for x in a for y in b]

def clausify(f):
    g = _skolem(_nnf(_elim(f)), {}, [])
    out = []
    for c in _cnf(g):
        c = list(dict.fromkeys(c))
        if any((not s, p, a) in c for s, p, a in c): continue          # tautology
        out.append(tuple(c))
    return out

def _walk(t, s):
    while isinstance(t, str) and t[:1] == '?' and t in s: t = s[t]
    return t

def _occurs(v, t, s):
    t = _walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(_occurs(v, x, s) for x in t[1:])

def unify(a, b, s):
    a, b = _walk(a, s), _walk(b, s)
    if a == b: return s
    if isinstance(a, str) and a[:1] == '?':
        return None if _occurs(a, b, s) else {**s, a: b}
    if isinstance(b, str) and b[:1] == '?':
        return None if _occurs(b, a, s) else {**s, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and len(a) == len(b) and a[0] == b[0]:
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def _apply(t, s):
    t = _walk(t, s)
    if isinstance(t, tuple): return (t[0],) + tuple(_apply(x, s) for x in t[1:])
    return t

def _rename(t, k):
    if isinstance(t, tuple): return (t[0],) + tuple(_rename(x, k) for x in t[1:])
    return f'{t}~{k}' if isinstance(t, str) and t[:1] == '?' else t

def _clause_key(c):
    names, out = {}, []
    def norm(t):
        if isinstance(t, tuple): return (t[0],) + tuple(norm(x) for x in t[1:])
        if isinstance(t, str) and t[:1] == '?': return names.setdefault(t, f'?{len(names)}')
        return t
    for s, p, a in sorted(c, key=lambda l: (l[1], l[0], repr(l[2]))):
        out.append((s, p, tuple(norm(x) for x in a)))
    return tuple(sorted(out, key=repr))

def _weight(c):
    def w(t): return 1 + (sum(w(x) for x in t[1:]) if isinstance(t, tuple) else 0)
    return sum(1 + sum(w(x) for x in a) for _, _, a in c)

def _resolvents(c1, c2, k):
    c2r = [(s, p, tuple(_rename(x, k) for x in a)) for s, p, a in c2]
    for i, (s1, p1, a1) in enumerate(c1):
        for j, (s2, p2, a2) in enumerate(c2r):
            if s1 == s2 or p1 != p2 or len(a1) != len(a2): continue
            sub = {}
            for x, y in zip(a1, a2):
                sub = unify(x, y, sub)
                if sub is None: break
            if sub is None: continue
            new = [(s, p, tuple(_apply(x, sub) for x in a)) for n, (s, p, a) in enumerate(c1) if n != i]
            new += [(s, p, tuple(_apply(x, sub) for x in a)) for n, (s, p, a) in enumerate(c2r) if n != j]
            yield tuple(dict.fromkeys(new))

def _factors(c):
    for i in range(len(c)):
        for j in range(i + 1, len(c)):
            if c[i][0] == c[j][0] and c[i][1] == c[j][1] and len(c[i][2]) == len(c[j][2]):
                sub = {}
                for x, y in zip(c[i][2], c[j][2]):
                    sub = unify(x, y, sub)
                    if sub is None: break
                if sub is not None:
                    yield tuple(dict.fromkeys((s, p, tuple(_apply(x, sub) for x in a)) for s, p, a in c))

def refute(axioms, sos, time_limit=5.0, max_clauses=30000, max_len=10):
    """set-of-support resolution; returns 'unsat' | 'sat' | 'unknown'"""
    t0, cnt, seen, usable, heap = time.time(), itertools.count(), set(), [], []
    def push(c):
        if any((not s, p, a) in c for s, p, a in c): return
        key = _clause_key(c)
        if key in seen or len(c) > max_len: return
        seen.add(key); heapq.heappush(heap, (_weight(c), next(cnt), c))
    usable = [tuple(c) for c in axioms]
    for c in sos: push(tuple(c))
    k = 0
    while heap:
        if time.time() - t0 > time_limit or len(seen) > max_clauses: return 'unknown'
        _, _, g = heapq.heappop(heap)
        if not g: return 'unsat'
        for f in _factors(g): push(f)
        for o in usable + [g]:
            k += 1
            for r in _resolvents(g, o, k):
                if not r: return 'unsat'
                push(r)
        usable.append(g)
    return 'sat'

def fol_entails(premises, conclusion, timeout=5.0):
    """premises: list[str] FOL formulas; conclusion: str. Returns 'True' | 'False' | 'Uncertain'."""
    P, skipped = [], 0
    for p in premises:
        try: P += clausify(FOLParser(p).parse())
        except Exception: skipped += 1
    conc = FOLParser(conclusion).parse()
    if refute(P, clausify(('not', conc)), timeout) == 'unsat': return 'True'
    if refute(P, clausify(conc), timeout) == 'unsat': return 'False'
    return 'Uncertain'

## 6. Causal-model reasoner (CounterBench-style counterfactuals)
CounterBench describes a deterministic causal graph in text (`X causes V1`, `X and V1 together cause V2`) and asks
*“Would Y occur if not X instead of X?”* (optionally with observations).  Internal representation: a Boolean structural model
(`together cause` = conjunction of parents; several causes of one effect = disjunction; parentless variables are exogenous).
Reasoning is classical: **abduction** (enumerate exogenous assignments consistent with the evidence), **intervention** (cut the
intervened variables' equations) and **prediction** (evaluate the model).  The query is *yes* iff the outcome holds in the consistent worlds.

*Note:* the exact released text format could not be inspected offline, so the patterns below are deliberately flexible.

In [7]:
def _vname(s):
    s = re.sub(r'^(?:that|the)\s+', '', s.strip().lower())
    return re.sub(r'[^a-z0-9_ \-]', '', s).strip()

def _literals(s):
    out = []
    for part in re.split(r'\s+and\s+|,\s*', s.strip().lower()):
        part = part.strip()
        if not part: continue
        neg = bool(re.match(r'^(not|no)\s+', part))
        part = re.sub(r'^(not|no)\s+', '', part)
        part = re.sub(r'\s+(?:occurs?|happens?|is true|is present)$', '', part)
        out.append((_vname(part), not neg))
    return out

def scm_parse(context):
    rules = []                                            # (target, [parents])
    text = re.sub(r'^(?:we know that|we know)\s+', '', context.strip().lower())
    for cl in re.split(r'[.;]\s*|,\s*(?:and\s+)?', text):
        cl = cl.strip()
        cl = re.sub(r'^(?:we know that|and)\s+', '', cl)
        if not cl: continue
        m = re.match(r'^(.+?)\s+together\s+(?:cause|causes)\s+(.+)$', cl)
        if m:
            rules.append((_vname(m.group(2)), [_vname(x) for x in re.split(r'\s+and\s+|,\s*', m.group(1))])); continue
        m = re.match(r'^(.+?)\s+(?:causes|cause|directly causes|leads to|has a direct effect on|directly affects)\s+(.+)$', cl)
        if m: rules.append((_vname(m.group(2)), [_vname(m.group(1))]))
    return rules

def scm_solve(context, question):
    rules = scm_parse(context)
    q = question.strip().lower()
    m = re.search(r'would\s+(.+?)\s*(?:still\s+)?(?:occur|happen|be true|exist|be present)?\s*if\s+(.+?)\s+instead of\s+(.+?)\s*\??\s*$', q)
    if not m or not rules: return None
    y_lit = _literals(m.group(1))[0]
    do, fact = _literals(m.group(2)), _literals(m.group(3))
    evidence = list(fact)
    for o in re.findall(r'observed\s+(?:that\s+)?(.+?)(?:[.?]|\bwould\b|$)', (context + ' ' + question).lower()):
        evidence += _literals(o)
    targets = defaultdict(list)
    for tgt, pa in rules: targets[tgt].append(pa)
    variables = set(targets) | {p for _, pa in rules for p in pa} | {v for v, _ in do + evidence} | {y_lit[0]}
    roots = sorted(v for v in variables if v not in targets)
    def evaluate(exo, fixed):
        val = dict(fixed)
        def get(v, stack=()):
            if v in val: return val[v]
            if v in stack: return False
            if v in exo and v not in targets: r = exo[v]
            else: r = any(all(get(p, stack + (v,)) for p in pa) for pa in targets.get(v, [])) if v in targets else exo.get(v, True)
            val[v] = r; return r
        for v in variables: get(v)
        return val
    outcomes = []
    for bits in itertools.product([True, False], repeat=len(roots)):
        exo = dict(zip(roots, bits))
        fv = evaluate(exo, {})
        if all(fv[v] == s for v, s in evidence):
            iv = evaluate(exo, {v: s for v, s in do})
            outcomes.append(iv[y_lit[0]] == y_lit[1])
    if not outcomes:
        exo = {r: True for r in roots}
        outcomes = [evaluate(exo, {v: s for v, s in do})[y_lit[0]] == y_lit[1]]
    return 'True' if sum(outcomes) * 2 >= len(outcomes) and any(outcomes) else 'False'

## 7. Unified reasoner
`Reasoner.solve_row(row)` picks the pipeline: first-order annotations present → resolution; causal-graph language → causal model;
otherwise the Horn pipeline (parser + forward/backward chaining).  The output is always the flag `True` or `False`.

In [8]:
def _first(row, keys, default=None):
    for k in keys:
        if k in row and row[k] is not None and str(row[k]).strip() != '': return row[k]
    return default

def norm_label(x):
    if isinstance(x, bool): return 'True' if x else 'False'
    s = str(x).strip().lower()
    if s in ('true', 'yes', '1', 'entailment', 'entailed', 'entails'): return 'True'
    if s in ('false', 'no', '0', 'not entailment', 'not_entailment', 'contradiction', 'not entailed'): return 'False'
    if s in ('uncertain', 'unknown', 'neutral'): return 'Uncertain'
    return s

class Reasoner:
    def __init__(self, **kw):
        c = dict(CONFIG); c.update(kw)
        self.mode, self.engine, self.fol_timeout = c['mode'], c['engine'], c['fol_timeout']
        self._cache, self.stats = {}, defaultdict(int)

    # ---- Horn pipeline
    def _model(self, context, want_prov=False):
        key = (context, self.mode, self.engine, want_prov)
        if key in self._cache: return self._cache[key]
        th = parse_theory(context)
        self.stats['sentences'] += len(split_sentences(context)); self.stats['unparsed'] += len(th.unparsed)
        facts, crules = compile_theory(th, self.mode)
        dom = sorted(th.domain)
        if self.engine == 'bc':
            m = ('bc', BackwardChainer(facts, crules, dom), facts, None)
        else:
            T, O, prov = solve_fc(facts, crules, dom, self.mode, want_prov)
            m = ('fc', (T, O), facts, prov)
        if len(self._cache) > 3000: self._cache.clear()
        self._cache[key] = m
        return m

    def horn(self, context, query, want_prov=False):
        kind, model, facts, prov = self._model(context, want_prov)
        lits = parse_query(query)
        if kind == 'bc':
            res = []
            for s, p, a in lits:
                if self.mode == 'cwa' or s: ok = model.prove(p, a) if s else (not model.prove(p, a))
                else: ok = model.prove('~' + p, a)
                res.append(ok)
            ok = all(res)
        else:
            ok = all(holds(model, l, self.mode) for l in lits)
        return ok, (lits, model, facts, prov)

    def explain(self, context, query):
        saved = self.engine; self.engine = 'fc'
        ok, (lits, model, facts, prov) = self.horn(context, query, want_prov=True)
        self.engine = saved
        lines = [f'Query: {query}  →  {ok}']
        for s, p, a in lits:
            if s and ok: lines += explain_fc(prov, facts, p, a, 1)
            elif not s and ok: lines.append(f'    {atom_str(p, a)} cannot be derived (negation as failure)')
            else: lines.append(f'    {lit_str((s, p, a))}: no derivation found → False')
        return '\n'.join(lines)

    # ---- unified entry points
    def solve_row(self, row, detail=False):
        try:
            return self._solve_row(row, detail)
        except Exception:                      # never crash a whole evaluation because of one malformed example
            self.stats['error'] += 1
            return 'False'

    def _solve_row(self, row, detail=False):
        ctx = _first(row, ['context', 'theory', 'premises', 'story', 'text', 'facts', 'background'], '')
        qst = _first(row, ['question', 'statement', 'conclusion', 'query', 'hypothesis', 'prompt'], '')
        if isinstance(ctx, (list, tuple)): ctx = '\n'.join(map(str, ctx))
        pf, cf = _first(row, ['premises-FOL', 'premises_FOL', 'premises_fol']), _first(row, ['conclusion-FOL', 'conclusion_FOL', 'conclusion_fol'])
        if pf and cf:
            try:
                prem = pf if isinstance(pf, (list, tuple)) else str(pf).split('\n')
                r = fol_entails([p for p in prem if p.strip()], str(cf), self.fol_timeout)
                self.stats['fol'] += 1
                return r if detail else ('True' if r == 'True' else 'False')
            except Exception:
                self.stats['fol_fail'] += 1
        low = (str(ctx) + ' ' + str(qst)).lower()
        if re.search(r'\bwould\b.*\bif\b.*\binstead of\b', low) and re.search(r'\bcauses?\b|\bdirect effect\b|\bleads to\b', low):
            r = scm_solve(str(ctx), str(qst))
            if r is not None:
                self.stats['scm'] += 1; return r
        ok, _ = self.horn(str(ctx), str(qst))
        self.stats['horn'] += 1
        return 'True' if ok else 'False'

    def solve(self, context, query):
        return self.solve_row({'context': context, 'question': query})

## 8. Data access
The notebook loads the three public datasets from Hugging Face (`datasets` library). Dataset IDs can be edited in `HF_IDS`;
if a dataset cannot be downloaded (e.g. no internet) the cell prints a message and continues. A synthetic RuleTaker-style
generator (with an *independent* grounding-based oracle) is provided **only as a correctness check of the engine**.

In [9]:
HF_IDS = {
    'RuleTaker':    ['tasksource/ruletaker', 'hitachi-nlp/ruletaker'],
    'CounterBench': ['CounterBench/CounterBench', 'Qiyuan-Chen/CounterBench'],
    'FOLIO':        ['yale-nlp/FOLIO', 'tasksource/folio'],
}

def load_hf(name):
    try:
        from datasets import load_dataset
    except Exception as e:
        print(f'[{name}] `datasets` not available: {e}'); return None
    for ds_id in HF_IDS[name]:
        try:
            ds = load_dataset(ds_id)
            out = {sp: [dict(r) for r in ds[sp]] for sp in ds.keys()}
            print(f'[{name}] loaded {ds_id}:', {k: len(v) for k, v in out.items()}, '| columns:', list(next(iter(out.values()))[0].keys()))
            return out
        except Exception as e:
            print(f'[{name}] could not load {ds_id}: {type(e).__name__}: {str(e)[:100]}')
    return None

def read_table(path):
    """csv / tsv / json / jsonl / parquet -> list of dict rows"""
    p = path.lower()
    if p.endswith('.parquet'):
        import pandas as pd; return pd.read_parquet(path).to_dict('records')
    if p.endswith('.jsonl'):
        return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    if p.endswith('.json'):
        d = json.load(open(path, encoding='utf-8')); return d if isinstance(d, list) else list(d.values())[0]
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f, delimiter='\t' if p.endswith('.tsv') else ','))

# ---- synthetic RuleTaker-style generator with an independent oracle (sanity check only)
NAMES = ['bob', 'anne', 'charlie', 'dave', 'erin', 'fiona', 'gary', 'harry']
ANIMALS = ['cat', 'dog', 'cow', 'bear', 'lion', 'tiger', 'mouse', 'rabbit', 'squirrel', 'bald eagle']
ATTRS = ['big', 'blue', 'red', 'green', 'kind', 'nice', 'rough', 'round', 'young', 'cold', 'smart', 'quiet']
RELS = ['likes', 'chases', 'eats', 'sees', 'visits', 'needs']

def _disp(e, first=False):
    if e in NAMES: return e.capitalize()
    return ('The ' if first else 'the ') + e

def gen_synthetic(n_theories=60, qs_per=6, seed=0, max_strata=3, size=1):
    rnd = random.Random(seed); rows = []
    for ti in range(n_theories):
        ents = rnd.sample(NAMES, rnd.randint(1, 3 + size)) + rnd.sample(ANIMALS, rnd.randint(1, 3 + size))
        attrs = rnd.sample(ATTRS, rnd.randint(5, 9))
        stratum = {a: rnd.randint(0, max_strata - 1) for a in attrs}
        facts_attr = {(a, e) for _ in range(rnd.randint(3, 7) * size) for a in [rnd.choice(attrs)] for e in [rnd.choice(ents)]}
        facts_rel = {(rnd.choice(RELS), rnd.choice(ents), rnd.choice(ents)) for _ in range(rnd.randint(1, 4) * size)}
        rules = []                                          # (body, head, text); body items: ('a',attr,neg) | ('r',rel,obj) ; head attr / or ground
        for _ in range(rnd.randint(3, 8) * size):
            h = rnd.choice(attrs); hs = stratum[h]
            pos_ok = [a for a in attrs if stratum[a] <= hs and a != h]
            neg_ok = [a for a in attrs if stratum[a] < hs]
            kind = rnd.choice(['if', 'if', 'all', 'rel', 'ground', 'not'])
            if kind in ('if', 'all') and pos_ok:
                b = rnd.sample(pos_ok, min(len(pos_ok), rnd.randint(1, 2)))
                body = [('a', x, False) for x in b]
                if kind == 'all':
                    text = ('All ' if rnd.random() < .5 else '') + ', '.join(b) + ' things are ' + h + '.'
                    text = text[0].upper() + text[1:]
                else:
                    text = 'If something is ' + ' and '.join(b) + f' then it is {h}.'
                rules.append((body, ('v', h), text))
            elif kind == 'not' and neg_ok and pos_ok:
                nb, pb = rnd.choice(neg_ok), rnd.choice(pos_ok)
                rules.append(([('a', pb, False), ('a', nb, True)], ('v', h), f'If something is {pb} and not {nb} then it is {h}.'))
            elif kind == 'rel':
                rel, obj = rnd.choice(RELS), rnd.choice(ents)
                rules.append(([('r', rel, obj)], ('v', h), f'If something {rel} {_disp(obj)} then it is {h}.'))
            elif kind == 'ground':
                rel, e1, e2 = rnd.choice(RELS), rnd.choice(ents), rnd.choice(ents)
                rules.append(([('g', rel, (e1, e2))], ('c', h, e2), f'If {_disp(e1)} {rel} {_disp(e2)} then {_disp(e2)} is {h}.'))
        # oracle: ground everything, naive fixpoint per stratum (independent of the main engine)
        attr_true = set(facts_attr)
        def body_ok(body, e):
            for it in body:
                if it[0] == 'a':
                    has = (it[1], e) in attr_true
                    if has == it[2]: return False
                elif it[0] == 'r':
                    if not any(r == it[1] and a == e and b == it[2] for (r, a, b) in facts_rel): return False
                elif it[0] == 'g':
                    if (it[1],) + it[2] not in {(r, a, b) for (r, a, b) in facts_rel}: return False
            return True
        for s in range(max_strata):
            changed = True
            while changed:
                changed = False
                for body, head, _ in rules:
                    if stratum[head[1]] != s: continue
                    for e in ents:
                        if head[0] == 'c' and e != head[2]: continue
                        if body_ok(body, e) and (head[1], e) not in attr_true:
                            attr_true.add((head[1], e)); changed = True
        lines = [f'{_disp(e, True)} is {a}.' for a, e in sorted(facts_attr)]
        lines += [f'{_disp(a, True)} {r} {_disp(b)}.' for r, a, b in sorted(facts_rel)]
        lines += [t for _, _, t in rules]
        rnd.shuffle(lines)
        ctx = ' '.join(lines)
        for _ in range(qs_per):
            e, a = rnd.choice(ents), rnd.choice(attrs); neg = rnd.random() < .3
            truth = ((a, e) in attr_true) != neg
            rows.append({'id': f'syn{ti}_{len(rows)}', 'context': ctx,
                         'question': f'{_disp(e, True)} is {"not " if neg else ""}{a}.', 'label': str(truth)})
    return rows

In [10]:
def evaluate(rows, reasoner, name='', verbose=True):
    t0 = time.perf_counter()
    preds = [reasoner.solve_row(r) for r in rows]
    dt = time.perf_counter() - t0
    gold = [norm_label(_first(r, ['label', 'answer', 'gold', 'target'])) for r in rows]
    correct = sum(p == g for p, g in zip(preds, gold))
    res = dict(n=len(rows), accuracy=100.0 * correct / max(1, len(rows)), seconds=dt, ms_per_query=1000 * dt / max(1, len(rows)))
    three = [g for g in gold if g == 'Uncertain']
    if verbose:
        print(f'{name:<34} n={res["n"]:<6} accuracy={res["accuracy"]:6.2f}%  time={dt:7.2f}s  ({res["ms_per_query"]:.2f} ms/query)')
    return res, preds, gold

## 9. Unit tests (hand-written, known answers)

In [11]:
def run_unit_tests():
    R = Reasoner(mode='cwa')
    ctx = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful."
    assert R.solve(ctx, "Bob is successful.") == 'True'                 # the assignment's example
    assert R.solve(ctx, "Bob is not successful.") == 'False'
    assert R.solve(ctx, "Bob is rich.") == 'False'
    ctx2 = ("The bald eagle likes the dog. The cat is red. If something likes the dog then it is blue. All red things are rough. "
            "If something is blue and rough then it is big. If something is not big and it is red then it is kind.")
    assert R.solve(ctx2, "The bald eagle is blue.") == 'True'
    assert R.solve(ctx2, "The cat is kind.") == 'True'                  # NAF: cat is not big
    assert R.solve(ctx2, "The cat is not big.") == 'True'
    assert R.solve(ctx2, "The bald eagle is kind.") == 'False'
    assert Reasoner(engine='bc').solve(ctx2, "The cat is kind.") == 'True'
    O = Reasoner(mode='owa')
    ctx3 = "Bob is big. Bob is not red. If someone is big then they are not kind. If someone is not red then they are nice. Charlie is young."
    assert O.solve(ctx3, "Bob is nice.") == 'True' and O.solve(ctx3, "Bob is not kind.") == 'True'
    assert O.solve(ctx3, "Charlie is nice.") == 'False'                 # Unknown -> False in binary output
    P = ["∀x (Dog(x) → Animal(x))", "Dog(rex)", "∀x (Cat(x) ⊕ Dog(x))", "Cat(tom)", "∀x (Animal(x) → ∃y (Needs(x,y) ∧ Food(y)))"]
    assert fol_entails(P, "Animal(rex)") == 'True' and fol_entails(P, "Dog(tom)") == 'False'
    assert fol_entails(P, "∃y (Needs(rex,y) ∧ Food(y))") == 'True' and fol_entails(P, "Animal(tom)") == 'Uncertain'
    assert scm_solve("We know that X causes V1, V1 causes V2.", "Would V2 occur if not X instead of X?") == 'False'
    print('all unit tests passed ✔')
run_unit_tests()

all unit tests passed ✔


## 10. Experiments on the public datasets
* RuleTaker: choose the negation semantics (`cwa` vs `owa`) on labelled public data and compare forward vs backward chaining.
* FOLIO: first-order resolution when `premises-FOL` is present (binary accuracy treats `Uncertain` as `False`; 3-way accuracy also reported).
* CounterBench: causal-model reasoner.

If a dataset cannot be downloaded the cell says so explicitly; a clearly-labelled **synthetic** check is always run as an engine sanity test.

In [12]:
EVAL_LIMIT = 5000       # max examples per dataset for the development experiments (None = all)
RESULTS, COMPARE, TRACES = {}, [], []
T_START = time.perf_counter()

def labelled_rows(d):
    if not d: return None, None
    for sp in ('validation', 'dev', 'test', 'train'):
        if sp in d and d[sp] and any(k in d[sp][0] and d[sp][0][k] is not None for k in ('label', 'answer')):
            rows = d[sp]
            if EVAL_LIMIT and len(rows) > EVAL_LIMIT: rows = random.Random(0).sample(rows, EVAL_LIMIT)
            return sp, rows
    return None, None

DATA = {name: load_hf(name) for name in HF_IDS}
if not any(DATA.values()):
    print('\n' + '!' * 78 + '\n  NO PUBLIC DATASET COULD BE DOWNLOADED in this run (no internet / Hugging Face blocked).\n'
          '  Only the synthetic engine-sanity check below is run. Re-run this notebook with internet\n'
          '  access (e.g. Google Colab) to obtain the real RuleTaker / CounterBench / FOLIO numbers.\n' + '!' * 78)

def three_way(rows, R):
    c3 = 0
    for r in rows:
        p = R.solve_row(r, detail=True); g = norm_label(_first(r, ['label', 'answer']))
        c3 += (p == g)
    return 100.0 * c3 / max(1, len(rows))

for name, d in DATA.items():
    sp, rows = labelled_rows(d)
    if rows is None: continue
    print(f'\n=== {name} (split "{sp}", {len(rows)} labelled examples) ===')
    if name == 'RuleTaker':
        best = None
        for mode in ('cwa', 'owa'):
            res, _, _ = evaluate(rows, Reasoner(mode=mode), f'RuleTaker  mode={mode} engine=fc')
            COMPARE.append((name, f'{mode} / forward chaining', res)); best = max(best or (res, mode), (res, mode), key=lambda x: x[0]['accuracy'])
        CONFIG['mode'] = best[1]; print('→ selected negation semantics:', CONFIG['mode'])
        sub = rows[:1000]
        res_bc, _, _ = evaluate(sub, Reasoner(mode=CONFIG['mode'], engine='bc'), f'RuleTaker  mode={CONFIG["mode"]} engine=bc (1000 ex.)')
        COMPARE.append((name, f'{CONFIG["mode"]} / backward chaining (first 1000)', res_bc))
    R = Reasoner(mode=CONFIG['mode'])
    res, preds, gold = evaluate(rows, R, f'{name} FINAL')
    res.update(split=sp, mode=CONFIG['mode'], engine='fc', parse_unparsed=R.stats['unparsed'], parse_sentences=R.stats['sentences'], source='public dataset')
    if name == 'FOLIO':
        res['three_way_accuracy'] = three_way(rows, Reasoner(mode=CONFIG['mode']))
        print(f'   FOLIO 3-way accuracy (True/False/Uncertain): {res["three_way_accuracy"]:.2f}%   pipeline counters: {dict(R.stats)}')
    RESULTS[name] = res
    COMPARE.append((name, 'final system', res))

# synthetic sanity check (independent oracle) — NOT a public-data result
print('\n=== Synthetic RuleTaker-style sanity check (independent grounding oracle) ===')
for size in (1, 3, 5):
    rows = gen_synthetic(120, 5, seed=100 + size, size=size)
    for eng in ('fc', 'bc'):
        res, _, _ = evaluate(rows, Reasoner(mode='cwa', engine=eng), f'synthetic size={size} engine={eng}')
        COMPARE.append((f'Synthetic (size {size})', 'forward chaining' if eng == 'fc' else 'backward chaining', res))
        if eng == 'fc': RESULTS[f'Synthetic size {size}'] = dict(res, source='synthetic sanity check', mode='cwa', engine='fc')

[RuleTaker] `datasets` not available: No module named 'datasets'
[CounterBench] `datasets` not available: No module named 'datasets'
[FOLIO] `datasets` not available: No module named 'datasets'

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  NO PUBLIC DATASET COULD BE DOWNLOADED in this run (no internet / Hugging Face blocked).
  Only the synthetic engine-sanity check below is run. Re-run this notebook with internet
  access (e.g. Google Colab) to obtain the real RuleTaker / CounterBench / FOLIO numbers.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

=== Synthetic RuleTaker-style sanity check (independent grounding oracle) ===
synthetic size=1 engine=fc         n=600    accuracy=100.00%  time=   0.05s  (0.08 ms/query)
synthetic size=1 engine=bc         n=600    accuracy=100.00%  time=   0.04s  (0.06 ms/query)
synthetic size=3 engine=fc         n=600    accuracy=100.00%  time=   0.16s  (0.27 ms/query)
synthetic size=3 en

## 11. Reasoning traces

In [13]:
R = Reasoner(mode='cwa')
ex_ctx = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful."
TRACES.append(('Assignment example', R.explain(ex_ctx, 'Bob is successful.')))
ex2 = ("The bald eagle likes the dog. The cat is red. If something likes the dog then it is blue. All red things are rough. "
       "If something is blue and rough then it is big. If something is not big and it is red then it is kind.")
TRACES.append(('Negation as failure', R.explain(ex2, 'The cat is kind.')))
TRACES.append(('Failed query', R.explain(ex2, 'The bald eagle is kind.')))
P_ = ["∀x (Dog(x) → Animal(x))", "∀x (Animal(x) → ∃y (Needs(x,y) ∧ Food(y)))", "Dog(rex)"]
TRACES.append(('First-order resolution', 'Premises: ' + ' ; '.join(P_) + '\nConclusion ∃y (Needs(rex,y) ∧ Food(y))  →  ' + fol_entails(P_, '∃y (Needs(rex,y) ∧ Food(y))') +
               '\n(refutation: clauses of the negated conclusion resolve with Skolemised premises to the empty clause)'))
for name, d in DATA.items():
    sp, rows = labelled_rows(d)
    if rows and name == 'RuleTaker':
        for r in rows[:2]:
            TRACES.append((f'RuleTaker example (gold {r.get("label")})', R.explain(str(_first(r, ['context'])), str(_first(r, ['question'])))))
for t, x in TRACES[:3]: print(f'--- {t} ---\n{x}\n')

--- Assignment example ---
Query: Bob is successful.  →  True
    successful(bob)   ⇐ "If someone is smart and kind, then they are successful"
        smart(bob)   ⇐ "If someone is young, then they are smart"
            young(bob)   [given fact]
        kind(bob)   [given fact]

--- Negation as failure ---
Query: The cat is kind.  →  True
    kind(cat)   ⇐ "If something is not big and it is red then it is kind"
        red(cat)   [given fact]
        not big(cat)   [cannot be derived → NAF]

--- Failed query ---
Query: The bald eagle is kind.  →  False
    kind(bald eagle): no derivation found → False



## 12. `submission.csv` generator
Set `TEST_PATH` to the private test file (csv / tsv / json / jsonl / parquet). Columns are detected automatically
(`id`, `context`/`premises`, `question`/`statement`/`conclusion`; FOL columns used if present). Output: `id,label` with `True`/`False`.

In [14]:
TEST_PATH = None        # e.g. '/content/private_test.csv'

def make_submission(rows, path='submission.csv', reasoner=None):
    R = reasoner or Reasoner(mode=CONFIG['mode'])
    t0 = time.perf_counter()
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f); w.writerow(['id', 'label'])
        for i, r in enumerate(rows):
            w.writerow([_first(r, ['id', 'ID', 'example_id', 'idx'], i), R.solve_row(r)])
    dt = time.perf_counter() - t0
    print(f'wrote {path}: {len(rows)} predictions in {dt:.2f}s')
    return dt

if TEST_PATH and os.path.exists(TEST_PATH):
    make_submission(read_table(TEST_PATH), 'submission.csv')
else:
    demo = gen_synthetic(40, 5, seed=999)
    print('TEST_PATH not set → writing a DEMO file from synthetic data (not a real submission).')
    make_submission(demo, 'submission_demo_synthetic.csv')

TEST_PATH not set → writing a DEMO file from synthetic data (not a real submission).
wrote submission_demo_synthetic.csv: 200 predictions in 0.01s


## 13. Final summary (accuracy and inference time)

In [15]:
print('=' * 78)
print('FINAL RESULTS')
print('=' * 78)
public = {k: v for k, v in RESULTS.items() if v.get('source') == 'public dataset'}
for k, v in RESULTS.items():
    print(f'{k:<26} [{v.get("source")}]  accuracy = {v["accuracy"]:.2f}%   inference time = {v["seconds"]:.2f}s  ({v["ms_per_query"]:.2f} ms/query)')
if public:
    n = sum(v['n'] for v in public.values())
    print(f'\nOVERALL (public datasets, micro-avg): accuracy = {sum(v["accuracy"]*v["n"] for v in public.values())/n:.2f}%  '
          f'total inference time = {sum(v["seconds"] for v in public.values()):.2f}s')
else:
    print('\n(No public dataset was available in this run: figures above are from the synthetic sanity check only.)')
print(f'Total notebook run time so far: {time.perf_counter() - T_START:.1f}s')

FINAL RESULTS
Synthetic size 1           [synthetic sanity check]  accuracy = 100.00%   inference time = 0.05s  (0.08 ms/query)
Synthetic size 3           [synthetic sanity check]  accuracy = 100.00%   inference time = 0.16s  (0.27 ms/query)
Synthetic size 5           [synthetic sanity check]  accuracy = 100.00%   inference time = 0.36s  (0.60 ms/query)

(No public dataset was available in this run: figures above are from the synthetic sanity check only.)
Total notebook run time so far: 1.0s


## 14. Report generator (`report.pdf`)
Builds the report required by the assignment (Times New Roman, justified, 1.5 line spacing) (representation, algorithms, experiments, reasoning traces) from the numbers
and traces produced by this very run, so the report always matches the notebook.

In [16]:
def build_report(path='report.pdf'):
    try:
        import reportlab
    except ImportError:
        import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'reportlab'], check=False)
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
    from reportlab.lib import colors
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Preformatted, KeepTogether
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont

    def find(names, dirs):
        for d in dirs:
            for n in names:
                p = os.path.join(d, n)
                if os.path.exists(p): return p
        return None
    extra_dirs = []
    try:
        import matplotlib; extra_dirs.append(os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf'))
    except Exception: pass
    sdirs = ['/usr/share/fonts/truetype/msttcorefonts', '/usr/share/fonts/truetype/liberation', '/usr/share/fonts/liberation',
             '/usr/share/fonts/truetype/liberation2', '/Library/Fonts', '/System/Library/Fonts/Supplemental', 'C:/Windows/Fonts']
    ddirs = ['/usr/share/fonts/truetype/dejavu', '/usr/share/fonts/dejavu'] + extra_dirs
    # Times New Roman (or its metric-compatible twin Liberation Serif)
    reg = find(['Times_New_Roman.ttf', 'times.ttf', 'Times New Roman.ttf', 'LiberationSerif-Regular.ttf'], sdirs)
    bold = find(['Times_New_Roman_Bold.ttf', 'timesbd.ttf', 'Times New Roman Bold.ttf', 'LiberationSerif-Bold.ttf'], sdirs)
    ital = find(['Times_New_Roman_Italic.ttf', 'timesi.ttf', 'Times New Roman Italic.ttf', 'LiberationSerif-Italic.ttf'], sdirs)
    bital = find(['Times_New_Roman_Bold_Italic.ttf', 'timesbi.ttf', 'Times New Roman Bold Italic.ttf', 'LiberationSerif-BoldItalic.ttf'], sdirs)
    if reg and bold and ital and bital:
        for n, p in (('TNR', reg), ('TNR-B', bold), ('TNR-I', ital), ('TNR-BI', bital)): pdfmetrics.registerFont(TTFont(n, p))
        pdfmetrics.registerFontFamily('TNR', normal='TNR', bold='TNR-B', italic='TNR-I', boldItalic='TNR-BI')
        serif, serif_b = 'TNR', 'TNR-B'
    else:
        serif, serif_b = 'Times-Roman', 'Times-Bold'
    dj, djm = find(['DejaVuSans.ttf'], ddirs), find(['DejaVuSansMono.ttf'], ddirs)
    if dj and djm:
        pdfmetrics.registerFont(TTFont('DJ', dj)); pdfmetrics.registerFont(TTFont('DJM', djm)); sym, mono = 'DJ', 'DJM'
    else:
        sym, mono = serif, 'Courier'
    SYMS = '∀∃∧∨⊕⇐∪⊤'                                   # glyphs missing from Times-style fonts
    def fx(t): return re.sub('([%s])' % SYMS, lambda m: f'<font name="{sym}">{m.group(1)}</font>', t)

    FS = 12; LEAD = 1.5 * FS                             # 12 pt text, 1.5 line spacing
    body = ParagraphStyle('body', fontName=serif, fontSize=FS, leading=LEAD, alignment=TA_JUSTIFY, spaceAfter=6)
    title = ParagraphStyle('title', fontName=serif_b, fontSize=20, leading=26, alignment=TA_CENTER, spaceAfter=4)
    sub = ParagraphStyle('sub', fontName=serif, fontSize=12, leading=LEAD, alignment=TA_CENTER, spaceAfter=10)
    h = ParagraphStyle('h', fontName=serif_b, fontSize=14, leading=21, alignment=TA_LEFT, spaceBefore=10, spaceAfter=4, keepWithNext=1)
    cap = ParagraphStyle('cap', parent=body, fontSize=10.5, leading=15.75, alignment=TA_CENTER, spaceBefore=3)
    warn = ParagraphStyle('warn', parent=body, backColor=colors.HexColor('#fff3cd'), borderColor=colors.HexColor('#e0a800'), borderWidth=.8, borderPadding=6, spaceBefore=6, spaceAfter=12)
    cell = ParagraphStyle('cell', fontName=serif, fontSize=10.5, leading=13, alignment=TA_LEFT)
    cellh = ParagraphStyle('cellh', parent=cell, fontName=serif_b)
    code = ParagraphStyle('code', fontName=mono, fontSize=8.2, leading=12.3, backColor=colors.HexColor('#f3f3f3'), borderPadding=5, spaceBefore=2, spaceAfter=10)
    P = lambda t, st=body: Paragraph(fx(t), st)
    esc = lambda t: str(t).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    M = lambda t: f'<font name="{mono}" size="10.5">{t}</font>'
    public = {k: v for k, v in RESULTS.items() if v.get('source') == 'public dataset'}

    st = [P('General-Purpose Classical Reasoning System', title),
          P('AI Assignment Report<br/><i>Natural-language knowledge base → internal representation → classical reasoning → True / False</i>', sub)]
    if not public:
        st.append(P('<b>Data note.</b> The public datasets (RuleTaker, CounterBench and FOLIO) could not be downloaded when this report was generated. '
                    'The experiments in Section 4 therefore consist solely of an engine-correctness check on synthetic data and must not be read as results on the public datasets. '
                    'Running the notebook with internet access regenerates this report with the real figures.', warn))

    st += [P('1. Overview', h),
           P('This report describes a reasoning system that decides whether a query is logically entailed by a knowledge base written in natural language. '
             'The system follows the required pipeline: the text is first translated into an internal logical representation, and the final True/False decision is produced '
             'exclusively by classical reasoning algorithms. No generative AI model is used at any stage of the pipeline.')]

    st += [P('2. Chosen Representation', h),
           P('Every sentence of the knowledge base is converted by a deterministic, template-based parser into function-free Horn logic. A literal is a triple of a sign, a predicate and its arguments. '
             f'A property sentence such as “Bob is young” becomes {M("young(bob)")}, and a relational sentence such as “The cat chases the dog” becomes {M("chase(cat, dog)")}, with verbs reduced to their base form. '
             f'Rules are expressed as Horn clauses with a universally quantified variable: “If someone is smart and kind, then they are successful” becomes {M("smart(?x) ∧ kind(?x) → successful(?x)")}. '
             'The same form is produced for generic statements such as “All red things are big” or “Big, red things are nice”. Conjunctive conclusions are split into separate rules, '
             'and a continuation such as “smart <i>and kind</i>” inherits the subject of the preceding condition.'),
           P('Negation is supported under two semantics, and the better one is selected on development data. Under the <b>closed-world</b> semantics, a negated body literal means that the corresponding fact '
             f'cannot be derived (negation as failure). Under the <b>open-world</b> semantics, a negated literal is an explicit, derivable fact, stored internally under the predicate {M("~p")}. '
             'Sentences that match no template are retained as zero-argument propositional atoms and are counted as “unparsed”, so that parsing coverage can be reported honestly.'),
           P('Two further internal forms are used when the data requires them. First-order formulas, such as the annotations that accompany FOLIO, are converted to clauses in conjunctive normal form with Skolem terms. '
             'Causal descriptions in the style of CounterBench are converted to a Boolean structural causal model, in which “together cause” denotes a conjunction of parents.')]

    st += [P('3. Reasoning Algorithms', h),
           P('<b>Forward chaining.</b> The default engine is a semi-naive bottom-up procedure. In every round each rule is joined with the fact base so that at least one body literal matches a newly derived fact; '
             'unification is performed by pattern matching on ground tuples. Every derived fact records the rule instance that produced it, which allows complete proofs to be printed. '
             'The resulting model is cached per knowledge base, so several queries about the same theory require only one inference.'),
           P('<b>Negation as failure.</b> For the closed-world semantics the engine uses the alternating fixpoint. An under-estimate and an over-estimate of the true facts are refined alternately until they no longer change. '
             'On stratified theories this yields the perfect model; otherwise it yields the well-founded model, and any undefined answer is returned as False.'),
           P('<b>Backward chaining.</b> As a goal-directed alternative, the system implements SLD resolution with a loop check on ancestor goals, a depth limit, a step budget and negation as failure. '
             'It is used to compare efficiency with forward chaining.'),
           P('<b>Resolution.</b> For first-order input the system eliminates the connectives →, ↔ and ⊕, converts to negation normal form, Skolemises existential quantifiers and distributes ∨ over ∧ to obtain clauses. '
             'It then runs set-of-support resolution with unification (including the occurs check), factoring, tautology and duplicate removal, and smallest-clause-first selection under a time limit. '
             'A query is entailed (True) when the premises together with the negated conclusion are unsatisfiable, and contradicted (False) when the premises together with the conclusion are unsatisfiable; '
             'otherwise it is Uncertain, which is reported as False in the binary output.'),
           P('<b>Causal-model reasoning.</b> Counterfactual queries are answered in three classical steps: abduction of the exogenous assignments consistent with the evidence, intervention on the variables that are set by the question, '
             'and prediction of the outcome in the modified model.')]

    st += [P('4. Experiments', h)]
    if public:
        st.append(P('The public datasets were used for development only. For RuleTaker, the negation semantics were chosen by comparing both options on labelled public data, '
                    'and forward chaining was compared with backward chaining on the same examples. The private test set was never used.'))
    else:
        st.append(P('Because the public datasets were unavailable, the system was verified on randomly generated RuleTaker-style theories containing facts, relations, negation and stratified rules. '
                    'The labels of these theories come from an independent, grounding-based oracle, so agreement of 100% is expected and confirms that the parser and both chaining engines are correct. '
                    'These figures say nothing about accuracy on the real benchmarks.'))
    rows = [['Dataset', 'Configuration', 'n', 'Accuracy (%)', 'Time (s)', 'ms / query']]
    for name, cfg, r in COMPARE:
        rows.append([name, cfg, str(r['n']), f'{r["accuracy"]:.2f}', f'{r["seconds"]:.2f}', f'{r["ms_per_query"]:.2f}'])
    tb = Table([[Paragraph(esc(c), cellh if i == 0 else cell) for c in row] for i, row in enumerate(rows)],
               colWidths=[3.2*cm, 5.6*cm, 1.4*cm, 2.3*cm, 1.9*cm, 1.9*cm], repeatRows=1)
    tb.setStyle(TableStyle([('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#e3e3e3')), ('GRID', (0, 0), (-1, -1), .5, colors.grey),
                            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'), ('TOPPADDING', (0, 0), (-1, -1), 3), ('BOTTOMPADDING', (0, 0), (-1, -1), 3)]))
    st += [Spacer(1, 4), tb, P('Table 1. Accuracy and inference time of each configuration.', cap)]
    extra = []
    for k, v in public.items():
        if v.get('parse_sentences'): extra.append(f'{k}: {v["parse_sentences"] - v["parse_unparsed"]} of {v["parse_sentences"]} context sentences matched a parsing template')
        if 'three_way_accuracy' in v: extra.append(f'{k}: three-way accuracy (True/False/Uncertain) of {v["three_way_accuracy"]:.2f}%')
    if CONFIG.get('mode'): extra.append(f'the final system uses the {"closed-world" if CONFIG["mode"] == "cwa" else "open-world"} negation semantics')
    st.append(P('; '.join(extra)[0].upper() + '; '.join(extra)[1:] + '.'))
    if not public:
        st.append(P('Forward chaining pays a one-off cost to build the model of each theory, whereas backward chaining is cheaper on large theories when only a single query is asked.'))

    st += [P('5. Examples of Reasoning Traces', h),
           P('The traces below are produced by the system itself. Each line shows a derived fact and the rule (quoted from the knowledge base) that justifies it, indented under the facts it depends on.')]
    leads = {'Assignment example': 'The example from the assignment is solved by two chained rule applications.',
             'Negation as failure': 'Here the condition “not big” holds because no derivation of “big” exists.',
             'Failed query': 'A query that cannot be derived is answered False.',
             'First-order resolution': 'A first-order entailment is proved by refutation.'}
    for t, x in TRACES[:5]:
        block = [P(f'<b>{esc(t)}.</b> {leads.get(t, "")}'), Preformatted(x, code)]
        st.append(KeepTogether(block))

    st += [P('6. Limitations', h),
           P('Coverage depends on the template parser. Free-form English, which is typical of the natural-language premises in FOLIO, is handled only through the first-order annotations when they are provided. '
             'Existential statements and rules with two distinct variables are not parsed, and the exact text format of CounterBench was matched with flexible patterns rather than verified against the released data. '
             'These cases are the main sources of expected errors, and extending the parser templates is the most direct way to improve accuracy.')]

    def footer(canvas, doc):
        canvas.saveState(); canvas.setFont(serif, 10); canvas.drawCentredString(A4[0] / 2, 1.3 * cm, str(doc.page)); canvas.restoreState()
    SimpleDocTemplate(path, pagesize=A4, leftMargin=2.54*cm, rightMargin=2.54*cm, topMargin=2.3*cm, bottomMargin=2.2*cm,
                      title='General-Purpose Classical Reasoning System').build(st, onFirstPage=footer, onLaterPages=footer)
    print('wrote', path, '| body font:', 'Times New Roman-compatible' if serif == 'TNR' else 'Times-Roman (built-in)')


build_report('report.pdf')

wrote report.pdf | body font: Times New Roman-compatible
